# Análise Financeira - ClearBank

Notebook com validação de transações, métricas mensais, detecção de suspeitas e exportação para JSON

In [1]:
import csv
import json
from datetime import datetime

LIMITE_SUSPEITO = 10000.00
ARQUIVO_CSV = 'transacoes.csv'

In [ ]:
def validar_transacao(linha):
    try:
        # id
        id_raw = linha.get('id', '').strip()
        id_val = int(id_raw)
        if id_val <= 0:
            return None

        # cliente_id
        cliente_id = linha.get('cliente_id', '').strip()
        if not cliente_id:
            return None

        # data
        data_str = linha.get('data', '').strip()
        try:
            data_obj = datetime.strptime(data_str, '%Y-%m-%d')
        except Exception:
            return None

        # tipo
        tipo = linha.get('tipo', '').strip().lower()
        if tipo not in ['credito', 'debito']:
            return None

        # valor
        valor_raw = linha.get('valor', '').strip()
        try:
            valor = float(valor_raw)
        except Exception:
            return None
        if valor <= 0:
            return None

        # OK
        return {
            'id': id_val,
            'data': data_obj,
            'data_str': data_str,
            'cliente_id': cliente_id,
            'tipo': tipo,
            'valor': valor,
            'descricao': linha.get('descricao', '').strip(),
            'categoria': linha.get('categoria', '').strip(),
            'mes': data_obj.strftime('%Y-%m')
        }
    except Exception:
        return None

In [ ]:
def ler_transacoes(arquivo=ARQUIVO_CSV):
    transacoes = []
    invalidas = 0
    total = 0
    try:
        with open(arquivo, 'r', encoding='utf-8') as f:
            reader = csv.DictReader(f)
            for linha in reader:
                total += 1
                trans = validar_transacao(linha)
                if trans:
                    transacoes.append(trans)
                else:
                    invalidas += 1
    except FileNotFoundError:
        print(f"Erro: arquivo '{arquivo}' não encontrado.")
        return [], 0, 0
    except Exception as e:
        print('Erro ao ler arquivo:', e)
        return [], 0, 0
    return transacoes, invalidas, total

In [ ]:
def gerar_relatorio(transacoes):
    if not transacoes:
        return {}, None, None, 0
    rel = {}
    datas = [t['data'] for t in transacoes]
    data_min = min(datas)
    data_max = max(datas)
    dias = (data_max - data_min).days
    for t in transacoes:
        mes = t['mes']
        if mes not in rel:
            rel[mes] = {'quantidade': 0, 'total_credito': 0.0, 'total_debito': 0.0, 'valores': []}
        rel[mes]['quantidade'] += 1
        rel[mes]['valores'].append(t['valor'])
        if t['tipo'] == 'credito':
            rel[mes]['total_credito'] += t['valor']
        else:
            rel[mes]['total_debito'] += t['valor']
    for mes in rel:
        vals = rel[mes]['valores']
        rel[mes]['saldo'] = rel[mes]['total_credito'] - rel[mes]['total_debito']
        rel[mes]['media'] = sum(vals) / len(vals) if vals else 0
        rel[mes]['maior_valor'] = max(vals) if vals else 0
        rel[mes]['menor_valor'] = min(vals) if vals else 0
        del rel[mes]['valores']
    return rel, data_min, data_max, dias

In [ ]:
def exibir_suspeitas(transacoes):
    def fmt_br(valor):
        return f"{valor:,.2f}".replace(',', '_').replace('.', ',').replace('_', '.')
    suspeitas = [t for t in transacoes if t['valor'] > LIMITE_SUSPEITO]
    print()
    print('='*70)
    print(' ' * 15 + 'TRANSAÇÕES SUSPEITAS (> R$ 10.000,00)')
    print('='*70)
    if suspeitas:
        for t in suspeitas:
            print(f"ID: {t['id']} | Cliente: {t['cliente_id']} | Data: {t['data_str']} | R$ {fmt_br(t['valor'])}")
    else:
        print('Nenhuma transação suspeita encontrada.')


In [ ]:
def exibir_relatorio(relatorio, data_min, data_max, dias, total_validas, total_invalidas, transacoes):
    def fmt_br(valor):
        return f"{valor:,.2f}".replace(',', '_').replace('.', ',').replace('_', '.')
    print()
    print('='*70)
    print(' ' * 15 + 'RELATÓRIO FINANCEIRO - ClearBank')
    print('='*70)
    if data_min and data_max:
        print(f"Período analisado: {data_min.strftime('%d/%m/%Y')} → {data_max.strftime('%d/%m/%Y')}")
        print(f"   Dias: {dias} dia(s)")
    print(f"\nResumo da limpeza:")
    print(f"   Total de linhas lidas: {total_validas + total_invalidas}")
    print(f"   Linhas válidas: {total_validas}")
    print(f"   Linhas inválidas: {total_invalidas}")
    print()
    print('='*70)
    print(' ' * 20 + 'RESUMO MENSAL')
    print('='*70)
    for mes in sorted(relatorio.keys()):
        r = relatorio[mes]
        print(f"\nMês: {mes}")
        print(f"  Transações: {r['quantidade']}")
        print(f"  Total crédito: R$ {fmt_br(r['total_credito'])}")
        print(f"  Total débito:  R$ {fmt_br(r['total_debito'])}")
        print(f"  Saldo:         R$ {fmt_br(r['saldo'])}")
        print(f"  Média:         R$ {fmt_br(r['media'])}")
        print(f"  Maior valor:   R$ {fmt_br(r['maior_valor'])}")
        print(f"  Menor valor:   R$ {fmt_br(r['menor_valor'])}")
    exibir_suspeitas(transacoes)


In [ ]:
def salvar_json(relatorio, total_validas, total_invalidas, arquivo='relatorio.json'):
    dados = {
        'gerado_em': datetime.now().strftime('%Y-%m-%d'),
        'total_transacoes_validas': total_validas,
        'total_transacoes_invalidas': total_invalidas,
        'resumo_mensal': {}
    }
    for mes in sorted(relatorio.keys()):
        r = relatorio[mes]
        dados['resumo_mensal'][mes] = {
            'quantidade': r['quantidade'],
            'total_credito': round(r['total_credito'], 2),
            'total_debito': round(r['total_debito'], 2),
            'saldo': round(r['saldo'], 2),
            'media': round(r['media'], 2),
            'maior_valor': round(r['maior_valor'], 2),
            'menor_valor': round(r['menor_valor'], 2)
        }
    try:
        with open(arquivo, 'w', encoding='utf-8') as f:
            json.dump(dados, f, ensure_ascii=False, indent=2)
        print(f"Relatório salvo em: {arquivo}")
    except Exception as e:
        print('Erro ao salvar JSON:', e)

In [ ]:
# EXECUÇÃO PRINCIPAL
print('Iniciando análise...')
transacoes, invalidas, total = ler_transacoes(ARQUIVO_CSV)
print(f'Total lidas: {total} | Válidas: {len(transacoes)} | Inválidas: {invalidas}')
if transacoes:
    relatorio, data_min, data_max, dias = gerar_relatorio(transacoes)
    exibir_relatorio(relatorio, data_min, data_max, dias, len(transacoes), invalidas, transacoes)
    salvar_json(relatorio, len(transacoes), invalidas)
else:
    print('Nenhuma transação válida encontrada. Verifique o CSV.')